In [3]:
# Extract the part after "Comm groups 0:" and process only until "Comm groups 1"
if "Comm groups 0:" in content:
    numbers_part = content.split("Comm groups 0:")[1].split("Comm groups 1:")[0]
    # Split the numbers and count them
    numbers = numbers_part.split()
    count = len(numbers)
    print(f"Count of numbers after 'Comm groups 0:': {count}")
    prev_num = numbers[0]
    transition = 0
    for n in numbers:
        if n != prev_num:
            transition += 1
            prev_num = n
    print(f"Number of transitions: {transition}")
else:
    print("No 'Comm groups 0:' found in the file.")


Count of numbers after 'Comm groups 0:': 44
Number of transitions: 4


In [3]:
import os
import re

# Path to the folder
folder_path = 'output_500ns_provision'

# Initialize a list to store the extracted numbers
extracted_numbers = []

# Iterate through all files in the folder
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path):
        with open(file_path, 'r') as file:
            for line in file:
                # Check if the line matches the pattern
                match = re.search(r'\[workload\] \[info\] sys\[0\] finished, (\d+) cycles, exposed communication (\d+) cycles\.', line)
                if match:
                    # Extract the two numbers
                    cycles, exposed_communication = map(int, match.groups())
                    extracted_numbers.append((cycles, exposed_communication))
                    print(f"Filename: {filename}, {cycles}, {exposed_communication}")

# Print the extracted numbers
# print(extracted_numbers)

Filename: debug_network_1000ms.yml.txt, 8506078782, 8506077537
Filename: debug_network_100ms.yml.txt, 4906078782, 4906077537
Filename: debug_network_10ms.yml.txt, 4546078782, 4546077537
Filename: debug_network_1ms.yml.txt, 4510078782, 4510077537
Filename: debug_network_500ms.yml.txt, 6506078782, 6506077537
Filename: debug_network_50ms.yml.txt, 4706078782, 4706077537
Filename: debug_network_5ms.yml.txt, 4526078782, 4526077537


In [2]:
from collections import Counter
import yaml

def find_rank_comm_change(rank):

    signature_list = [f"RANK: {rank} Issuing collective CG(id=", f"RANK: {rank} Issuing SEND", f"RANK: {rank} Issuing RECV"]
    comm_group_list = [1, -1, -1] if rank == 0 or rank == 1 else [2, -1, -1]

    def find_signature_idx(signature, line):
        for i, sig in enumerate(signature_list):
            if sig in line:
                return i
        return -1

    # Path to the file
    file_path = 'debug_r.txt'

    # Initialize a counter to store occurrences
    comm_start_idx_map = {i: [] for i in comm_group_list}
    global_idx = 0

    # Read the file and process lines
    with open(file_path, 'r') as file:
        prev_group_idx = -100
        for line in file:
            signature_idx = find_signature_idx(signature_list, line)
            group_idx = comm_group_list[signature_idx] if signature_idx != -1 else None
            if signature_idx == -1:
                continue
            else:
                if group_idx != prev_group_idx:
                    comm_start_idx_map[group_idx].append(global_idx)
                    prev_group_idx = group_idx
                
                global_idx += 1

    for idx, sig in enumerate(signature_list):
        print(f"Signature: {sig}, group: {comm_group_list[idx]}, start indices: {comm_start_idx_map[comm_group_list[idx]]}")

    # Structure the data for YAML
    data = {rank: {group_idx: indices for group_idx, indices in comm_start_idx_map.items() if indices}}

    # Path to the YAML file
    yaml_file_path = 'rank_comm_groups.yaml'

    # Write the data to the YAML file
    with open(yaml_file_path, 'a') as yaml_file:
        yaml.dump(data, yaml_file, default_flow_style=False)

    print(f"Data for rank {rank} written to {yaml_file_path}")

for rank in range(4):
    find_rank_comm_change(rank)
    print()


Signature: RANK: 0 Issuing collective CG(id=, group: 1, start indices: [2]
Signature: RANK: 0 Issuing SEND, group: -1, start indices: [0]
Signature: RANK: 0 Issuing RECV, group: -1, start indices: [0]
Data for rank 0 written to rank_comm_groups.yaml

Signature: RANK: 1 Issuing collective CG(id=, group: 1, start indices: [2]
Signature: RANK: 1 Issuing SEND, group: -1, start indices: [0]
Signature: RANK: 1 Issuing RECV, group: -1, start indices: [0]
Data for rank 1 written to rank_comm_groups.yaml

Signature: RANK: 2 Issuing collective CG(id=, group: 2, start indices: [1]
Signature: RANK: 2 Issuing SEND, group: -1, start indices: [0, 42]
Signature: RANK: 2 Issuing RECV, group: -1, start indices: [0, 42]
Data for rank 2 written to rank_comm_groups.yaml

Signature: RANK: 3 Issuing collective CG(id=, group: 2, start indices: [1]
Signature: RANK: 3 Issuing SEND, group: -1, start indices: [0, 42]
Signature: RANK: 3 Issuing RECV, group: -1, start indices: [0, 42]
Data for rank 3 written to ran